# Quantum SAM: Skin Lesion Segmentation

**ISIC 2018 Task 1 — Hybrid Segment Anything Model with Quantum Channel Attention**

This notebook implements a hybrid architecture that combines Meta's Segment Anything Model (SAM) with a Quantum Channel Attention (QCA) module for skin lesion segmentation.

## Architecture

| Component | Status | Parameters |
|---|---|---|
| SAM ViT-B image encoder (frozen blocks) | Frozen | 74.7M |
| SAM ViT-B image encoder (last 2 blocks + neck) | Trainable | 15.0M |
| Quantum Channel Attention | Trainable | ~4K |
| SAM mask decoder | Trainable | 4.1M |
| **Total trainable** | | **~19M** |

## Why this approach

SAM is pretrained on 11M+ images, so its image encoder produces strong general-purpose features even for medical images outside its training distribution. Freezing most of the encoder reduces the risk of overfitting on the relatively small ISIC 2018 dataset (2,594 images). The Quantum Channel Attention module re-weights the 256-dim image embeddings before they reach the mask decoder, providing a learnable channel-wise gating mechanism with very few parameters.

The bounding-box prompt is computed automatically from the ground-truth mask during training. At inference time, prompts can be either bounding boxes or center points.

## 1. Setup

In [ ]:
# 1.1 - Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 1.2 - Install dependencies
!pip install segmentation_models_pytorch albumentations -q
!pip install pennylane -q
!pip install git+https://github.com/facebookresearch/segment-anything.git -q

In [ ]:
# 1.3 - Download SAM ViT-B weights
import os
import urllib.request

sam_checkpoint = 'sam_vit_b_01ec64.pth'
if not os.path.exists(sam_checkpoint):
    print("Downloading SAM ViT-B (~375 MB)...")
    url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
    urllib.request.urlretrieve(url, sam_checkpoint)
    print("SAM ViT-B downloaded.")
else:
    print("SAM ViT-B already present.")

print(f"  File size: {os.path.getsize(sam_checkpoint) / 1e6:.0f} MB")

In [ ]:
# 1.4 - Configure Kaggle API
from google.colab import files
print("Upload your kaggle.json:")
files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!pip uninstall kaggle kagglesdk -y -q
!pip install kaggle -q

In [ ]:
# 1.5 - Download ISIC 2018 dataset
for d in ['data/raw', 'models/best',
          'outputs/visualizations', 'outputs/predictions']:
    os.makedirs(d, exist_ok=True)

!kaggle datasets download -d tschandl/isic2018-challenge-task1-data-segmentation -p data/raw/ --unzip
print("ISIC 2018 dataset ready.")

## 2. Configuration and Imports

In [ ]:
# 2.1 - Imports
import os, cv2, random, math, time, copy, shutil
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# SAM
from segment_anything import sam_model_registry, SamPredictor
from segment_anything.modeling import Sam

# Quantum
import pennylane as qml

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda')
print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0)})")
print(f"PennyLane: {qml.__version__}")

In [ ]:
# 2.2 - Configuration
CONFIG = {
    # Data
    'seg_images': 'data/raw/ISIC2018_Task1-2_Training_Input',
    'seg_masks':  'data/raw/ISIC2018_Task1_Training_GroundTruth',

    # Model
    'sam_checkpoint': 'sam_vit_b_01ec64.pth',
    'sam_type':       'vit_b',
    'image_size':     1024,         # SAM native resolution
    'unfreeze_blocks': 2,           # last N transformer blocks made trainable

    # Training
    'batch_size':      4,
    'epochs':          80,
    'lr':              1e-4,        # decoder learning rate
    'encoder_lr':      5e-6,        # encoder learning rate (much lower)
    'weight_decay':    1e-4,
    'warmup_epochs':   5,
    'early_stopping':  15,

    # Loss
    'focal_gamma':    0.75,
    'tversky_alpha':  0.3,
    'tversky_beta':   0.7,

    # Quantum
    'n_qubits':        8,
    'q_layers':        2,
    'q_lr_multiplier': 5.0,         # quantum params train faster
    'q_warmup_epochs': 2,
}

print("Configuration loaded.")
print(f"  SAM: {CONFIG['sam_type']} @ {CONFIG['image_size']}px")
print(f"  Quantum: {CONFIG['n_qubits']} qubits, {CONFIG['q_layers']} layers")
print(f"  Epochs: {CONFIG['epochs']}, LR: {CONFIG['lr']}")

## 3. Dataset

The dataset class returns `(image, mask, bbox, center)` tuples. Bounding boxes and center points are computed from the ground-truth mask and used as prompts for SAM. Train/Val/Test split is 70/15/15 with seed=42, identical to the U-Net baseline.

In [ ]:
# 3.1 - SAM-compatible dataset
class SAMSkinDataset(Dataset):
    """
    Dataset for SAM-based segmentation.

    SAM expects 1024x1024 RGB input. The mask decoder produces 256x256 output,
    so ground-truth masks are resized to match. Prompts (bbox or center point)
    are derived from the ground-truth mask in the 256x256 mask space.
    """
    def __init__(self, image_dir, mask_dir, file_list, image_size=1024,
                 transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.file_list = file_list
        self.image_size = image_size
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_name = self.file_list[idx]
        mask_name = img_name.replace('.jpg', '_segmentation.png')

        image = cv2.imread(os.path.join(self.image_dir, img_name))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(
            os.path.join(self.mask_dir, mask_name),
            cv2.IMREAD_GRAYSCALE
        )
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug['image']
            mask = aug['mask']

        # Resize image to SAM's native resolution
        image = cv2.resize(image, (self.image_size, self.image_size))
        # Mask resized to SAM's output resolution
        mask = cv2.resize(
            mask, (256, 256),
            interpolation=cv2.INTER_NEAREST
        )

        # Compute prompts from mask
        bbox = self._get_bbox(mask)
        center = self._get_center(mask)

        # To tensor
        image = torch.tensor(
            image.transpose(2, 0, 1), dtype=torch.float32
        ) / 255.0
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        bbox = torch.tensor(bbox, dtype=torch.float32)
        center = torch.tensor(center, dtype=torch.float32)

        return image, mask, bbox, center

    def _get_bbox(self, mask):
        """Compute bounding box [x1, y1, x2, y2] from mask."""
        rows = np.any(mask > 0, axis=1)
        cols = np.any(mask > 0, axis=0)
        if not np.any(rows):
            return np.array([0, 0, 255, 255], dtype=np.float32)
        y1, y2 = np.where(rows)[0][[0, -1]]
        x1, x2 = np.where(cols)[0][[0, -1]]
        pad = 5
        y1 = max(0, y1 - pad)
        x1 = max(0, x1 - pad)
        y2 = min(255, y2 + pad)
        x2 = min(255, x2 + pad)
        return np.array([x1, y1, x2, y2], dtype=np.float32)

    def _get_center(self, mask):
        """Compute center point of mask."""
        if mask.sum() == 0:
            return np.array([128, 128], dtype=np.float32)
        ys, xs = np.where(mask > 0)
        return np.array([xs.mean(), ys.mean()], dtype=np.float32)

In [ ]:
# 3.2 - Augmentation pipeline
def get_sam_transforms(mode='train'):
    if mode == 'train':
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.1, scale_limit=0.15,
                rotate_limit=30, border_mode=cv2.BORDER_REFLECT_101,
                p=0.5
            ),
            A.OneOf([
                A.RandomBrightnessContrast(
                    brightness_limit=0.2, contrast_limit=0.2, p=1
                ),
                A.HueSaturationValue(
                    hue_shift_limit=15, sat_shift_limit=20,
                    val_shift_limit=15, p=1
                ),
                A.CLAHE(clip_limit=3.0, p=1),
            ], p=0.5),
            A.OneOf([
                A.GaussNoise(var_limit=(10, 40), p=1),
                A.GaussianBlur(blur_limit=(3, 5), p=1),
            ], p=0.2),
        ])
    return None

In [ ]:
# 3.3 - Train/Val/Test split (matched to U-Net baseline)
def split_dataset(image_dir, train_ratio=0.70, val_ratio=0.15, seed=42):
    random.seed(seed)
    images = sorted([
        f for f in os.listdir(image_dir) if f.endswith('.jpg')
    ])
    random.shuffle(images)
    n = len(images)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    return (images[:n_train],
            images[n_train:n_train+n_val],
            images[n_train+n_val:])

train_files, val_files, test_files = split_dataset(CONFIG['seg_images'])
print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

## 4. Model: Quantum SAM

**Strategy**

1. SAM ViT-B image encoder is loaded with pretrained weights. Most of the encoder is frozen; only the last 2 transformer blocks and the neck layer remain trainable.
2. The encoder output (256-dim image embeddings) is passed through a Quantum Channel Attention module that produces a per-channel gating signal.
3. The gated embeddings are fed to SAM's mask decoder, which is fully fine-tuned.
4. The bounding box prompt (derived from the ground-truth mask) drives the prompt encoder during training.

In [ ]:
# 4.1 - Quantum Channel Attention
class QuantumChannelAttention(nn.Module):
    """
    Channel attention with a parameterized quantum circuit.

    Pipeline:
      [B, C, H, W]
        -> global average pool to [B, C]
        -> linear compression to [B, n_qubits] with tanh
        -> quantum circuit per sample -> [B, n_qubits]
        -> linear expansion to [B, C] with sigmoid
        -> reshape to [B, C, 1, 1] and multiply with input
    """
    def __init__(self, in_channels, n_qubits=8, q_layers=2):
        super().__init__()
        self.in_channels = in_channels
        self.n_qubits = n_qubits
        self.q_layers = q_layers

        self.compress = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_channels, n_qubits),
            nn.Tanh()
        )

        self.dev = qml.device('default.qubit', wires=n_qubits)
        self.q_params = nn.Parameter(
            0.01 * torch.randn(q_layers, n_qubits, 3)
        )

        @qml.qnode(self.dev, interface='torch', diff_method='backprop')
        def quantum_circuit(inputs, weights):
            for layer_idx in range(q_layers):
                # Data encoding via RY rotations
                for i in range(n_qubits):
                    qml.RY(inputs[i] * np.pi, wires=i)
                # Variational rotations
                for i in range(n_qubits):
                    qml.Rot(
                        weights[layer_idx, i, 0],
                        weights[layer_idx, i, 1],
                        weights[layer_idx, i, 2],
                        wires=i
                    )
                # Ring entanglement
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
                qml.CNOT(wires=[n_qubits - 1, 0])
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.quantum_circuit = quantum_circuit

        self.expand = nn.Sequential(
            nn.Linear(n_qubits, in_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        B = x.shape[0]
        compressed = self.compress(x)
        q_outputs = []
        for i in range(B):
            q_out = self.quantum_circuit(compressed[i], self.q_params)
            q_outputs.append(torch.stack(q_out))
        q_out = torch.stack(q_outputs).float()
        gate = self.expand(q_out).unsqueeze(-1).unsqueeze(-1)
        return x * gate

# Sanity check
qca = QuantumChannelAttention(256, n_qubits=4, q_layers=2)
test_input = torch.randn(2, 256, 8, 8)
test_output = qca(test_input)
print(f"QCA forward pass: {test_input.shape} -> {test_output.shape}")
del qca, test_input, test_output

In [ ]:
# 4.2 - Quantum SAM wrapper
class QuantumSAM(nn.Module):
    """
    SAM + Quantum Channel Attention.

    Frozen:    most of SAM's image encoder
    Trainable: last 2 encoder blocks, neck, QCA, mask decoder
    """
    def __init__(self, sam_checkpoint, sam_type='vit_b',
                 n_qubits=8, q_layers=2, unfreeze_blocks=2):
        super().__init__()

        # Load SAM with pretrained weights
        self.sam = sam_model_registry[sam_type](checkpoint=sam_checkpoint)

        # Freeze the entire image encoder
        for param in self.sam.image_encoder.parameters():
            param.requires_grad = False

        # Unfreeze last N transformer blocks
        for param in self.sam.image_encoder.blocks[-unfreeze_blocks:].parameters():
            param.requires_grad = True
        # Unfreeze neck
        for param in self.sam.image_encoder.neck.parameters():
            param.requires_grad = True

        # Quantum attention on 256-dim SAM embeddings
        self.quantum_attention = QuantumChannelAttention(
            in_channels=256,
            n_qubits=n_qubits,
            q_layers=q_layers
        )

        # Report parameter counts
        encoder_frozen = sum(
            p.numel() for p in self.sam.image_encoder.parameters()
            if not p.requires_grad
        )
        encoder_trainable = sum(
            p.numel() for p in self.sam.image_encoder.parameters()
            if p.requires_grad
        )
        decoder_params = sum(
            p.numel() for p in self.sam.mask_decoder.parameters()
            if p.requires_grad
        )
        quantum_params = sum(
            p.numel() for p in self.quantum_attention.parameters()
        )
        total_trainable = encoder_trainable + decoder_params + quantum_params

        print(f"  SAM encoder (frozen):    {encoder_frozen:,}")
        print(f"  SAM encoder (trainable): {encoder_trainable:,}")
        print(f"  SAM decoder (trainable): {decoder_params:,}")
        print(f"  Quantum (trainable):     {quantum_params:,}")
        print(f"  Total trainable:         {total_trainable:,}")

    def forward(self, images, boxes=None, points=None):
        """
        Args:
            images: [B, 3, 1024, 1024]
            boxes:  [B, 4] bbox prompts (x1,y1,x2,y2 in 256-space)
            points: [B, 2] center point prompts (in 256-space)

        Returns:
            masks: [B, 1, 256, 256] logits
        """
        B = images.shape[0]

        # Image encoding
        image_embeddings = self.sam.image_encoder(images)
        # [B, 256, 64, 64]

        # Quantum channel attention
        image_embeddings = self.quantum_attention(image_embeddings)

        # Per-sample mask decoding
        all_masks = []
        for i in range(B):
            if boxes is not None:
                # Scale 256-space bbox to 1024-space for prompt encoder
                box_1024 = boxes[i:i+1] * 4.0
                box_torch = box_1024.unsqueeze(0).to(images.device)
                sparse_emb, dense_emb = self.sam.prompt_encoder(
                    points=None, boxes=box_torch, masks=None
                )
            elif points is not None:
                pt_1024 = points[i:i+1] * 4.0
                pt_torch = pt_1024.unsqueeze(0).to(images.device)
                pt_labels = torch.ones(
                    1, 1, dtype=torch.long, device=images.device
                )
                sparse_emb, dense_emb = self.sam.prompt_encoder(
                    points=(pt_torch, pt_labels), boxes=None, masks=None
                )
            else:
                sparse_emb, dense_emb = self.sam.prompt_encoder(
                    points=None, boxes=None, masks=None
                )

            low_res_masks, iou_preds = self.sam.mask_decoder(
                image_embeddings=image_embeddings[i:i+1],
                image_pe=self.sam.prompt_encoder.get_dense_pe(),
                sparse_prompt_embeddings=sparse_emb,
                dense_prompt_embeddings=dense_emb,
                multimask_output=False,
            )
            all_masks.append(low_res_masks.squeeze(0))

        return torch.stack(all_masks)

In [ ]:
# 4.3 - Build model
model = QuantumSAM(
    sam_checkpoint=CONFIG['sam_checkpoint'],
    sam_type=CONFIG['sam_type'],
    n_qubits=CONFIG['n_qubits'],
    q_layers=CONFIG['q_layers'],
    unfreeze_blocks=CONFIG['unfreeze_blocks']
).to(DEVICE)

print("Quantum SAM ready.")

In [ ]:
# 4.4 - Loss: Focal Tversky + BCE + Lovasz
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def forward(self, pred, target):
        pred_sig = torch.sigmoid(pred)
        smooth = 1e-6
        TP = (pred_sig * target).sum(dim=(2, 3))
        FP = (pred_sig * (1 - target)).sum(dim=(2, 3))
        FN = ((1 - pred_sig) * target).sum(dim=(2, 3))
        tversky = (TP + smooth) / (
            TP + self.alpha * FP + self.beta * FN + smooth
        )
        return torch.pow(1 - tversky, self.gamma).mean()


def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_hinge_flat(logits, labels):
    if len(labels) == 0:
        return logits.sum() * 0.0
    signs = 2.0 * labels.float() - 1.0
    errors = 1.0 - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    gt_sorted = labels[perm]
    grad = lovasz_grad(gt_sorted)
    return torch.dot(F.relu(errors_sorted), grad)


class CombinedLoss(nn.Module):
    """Weighted combination of Focal Tversky, BCE, and Lovasz hinge."""
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75):
        super().__init__()
        self.focal_tversky = FocalTverskyLoss(alpha, beta, gamma)
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred, target):
        ft = self.focal_tversky(pred, target)
        bce = self.bce(pred, target)
        lov = lovasz_hinge_flat(pred.view(-1), target.view(-1))
        return 0.35 * ft + 0.30 * bce + 0.35 * lov

print("CombinedLoss ready.")

## 5. Training

In [ ]:
# 5.1 - Train and evaluation functions
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    # Keep frozen encoder in eval mode (BatchNorm/Dropout)
    model.sam.image_encoder.eval()
    total_loss = 0

    for images, masks, boxes, centers in tqdm(loader, desc='  Train', leave=False):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        boxes = boxes.to(DEVICE)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            pred_masks = model(images, boxes=boxes)
            loss = criterion(pred_masks, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0
        )
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_iou, total_dice = 0, 0, 0

    with torch.no_grad():
        for images, masks, boxes, centers in tqdm(loader, desc='  Val', leave=False):
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)
            boxes = boxes.to(DEVICE)

            with torch.cuda.amp.autocast():
                pred_masks = model(images, boxes=boxes)
                loss = criterion(pred_masks, masks)
            total_loss += loss.item()

            pred_bin = (torch.sigmoid(pred_masks) > 0.5).float()
            inter = (pred_bin * masks).sum()
            union = (pred_bin + masks).clamp(0, 1).sum()
            total_iou += ((inter + 1e-6) / (union + 1e-6)).item()
            total_dice += (
                (2 * inter + 1e-6) /
                (pred_bin.sum() + masks.sum() + 1e-6)
            ).item()

    n = len(loader)
    return total_loss / n, total_iou / n, total_dice / n

In [ ]:
# 5.2 - Quantum freeze/unfreeze helper
def set_quantum_freeze(model, freeze=True):
    count = 0
    for name, param in model.named_parameters():
        if 'quantum' in name or 'q_params' in name:
            param.requires_grad = not freeze
            count += 1
    state = "frozen" if freeze else "unfrozen"
    print(f"  Quantum parameters {state} ({count} tensors)")

In [ ]:
# 5.3 - DataLoaders
train_ds = SAMSkinDataset(
    CONFIG['seg_images'], CONFIG['seg_masks'], train_files,
    image_size=CONFIG['image_size'],
    transform=get_sam_transforms('train')
)
val_ds = SAMSkinDataset(
    CONFIG['seg_images'], CONFIG['seg_masks'], val_files,
    image_size=CONFIG['image_size'],
    transform=None
)
train_loader = DataLoader(
    train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=4, pin_memory=True
)
print(f"Train: {len(train_ds)} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds)} samples ({len(val_loader)} batches)")

In [ ]:
# 5.4 - Training loop with quantum warmup
criterion = CombinedLoss(
    alpha=CONFIG['tversky_alpha'],
    beta=CONFIG['tversky_beta'],
    gamma=CONFIG['focal_gamma']
)
scaler = torch.cuda.amp.GradScaler()

def build_optimizer(model, encoder_lr, decoder_lr, quantum_lr, weight_decay):
    enc_p, dec_p, q_p = [], [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'quantum' in name or 'q_params' in name:
            q_p.append(param)
        elif 'image_encoder' in name:
            enc_p.append(param)
        else:
            dec_p.append(param)
    return torch.optim.AdamW([
        {'params': enc_p, 'lr': encoder_lr},
        {'params': dec_p, 'lr': decoder_lr},
        {'params': q_p, 'lr': quantum_lr},
    ], weight_decay=weight_decay), enc_p, dec_p, q_p

optimizer, encoder_params, decoder_params, quantum_params = build_optimizer(
    model,
    encoder_lr=CONFIG['encoder_lr'],
    decoder_lr=CONFIG['lr'],
    quantum_lr=CONFIG['lr'] * CONFIG['q_lr_multiplier'],
    weight_decay=CONFIG['weight_decay']
)

def lr_lambda(epoch):
    if epoch < CONFIG['warmup_epochs']:
        return (epoch + 1) / CONFIG['warmup_epochs']
    progress = (epoch - CONFIG['warmup_epochs']) / max(
        1, CONFIG['epochs'] - CONFIG['warmup_epochs']
    )
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_dice': [], 'lr': []}
best_iou = 0
patience = 0

print("Starting Quantum SAM training")
print(f"  Encoder params: {sum(p.numel() for p in encoder_params):,} (LR: {CONFIG['encoder_lr']})")
print(f"  Decoder params: {sum(p.numel() for p in decoder_params):,} (LR: {CONFIG['lr']})")
print(f"  Quantum params: {sum(p.numel() for p in quantum_params):,} (LR: {CONFIG['lr'] * CONFIG['q_lr_multiplier']})")
print(f"  Epochs: {CONFIG['epochs']}")
print("=" * 70)

for epoch in range(CONFIG['epochs']):
    # Quantum warmup: freeze QCA for first q_warmup_epochs, then unfreeze
    if epoch < CONFIG['q_warmup_epochs']:
        set_quantum_freeze(model, freeze=True)
    elif epoch == CONFIG['q_warmup_epochs']:
        set_quantum_freeze(model, freeze=False)
        # Rebuild optimizer with the new requires_grad state
        optimizer, _, _, _ = build_optimizer(
            model,
            encoder_lr=CONFIG['encoder_lr'],
            decoder_lr=CONFIG['lr'],
            quantum_lr=CONFIG['lr'] * CONFIG['q_lr_multiplier'],
            weight_decay=CONFIG['weight_decay']
        )
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        print("  Quantum warmup complete")

    current_lr = optimizer.param_groups[0]['lr']

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    val_loss, val_iou, val_dice = evaluate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    history['val_dice'].append(val_dice)
    history['lr'].append(current_lr)

    msg = (
        f"  E{epoch+1:02d}/{CONFIG['epochs']} | "
        f"T:{train_loss:.4f} V:{val_loss:.4f} | "
        f"IoU:{val_iou:.4f} D:{val_dice:.4f} | "
        f"LR:{current_lr:.6f}"
    )

    if val_iou > best_iou:
        best_iou = val_iou
        patience = 0
        torch.save({
            'model_state': {
                k: v for k, v in model.state_dict().items()
                if 'image_encoder' not in k or any(
                    f'blocks.{i}' in k for i in range(11, 12)
                ) or 'neck' in k
            },
            'best_iou': best_iou,
            'epoch': epoch,
        }, 'models/best/best_quantum_sam.pth')
        print(f"{msg}  [best]")
    else:
        patience += 1
        print(f"{msg}  ({patience}/{CONFIG['early_stopping']})")
        if patience >= CONFIG['early_stopping']:
            print(f"  Early stopping triggered at epoch {epoch+1}")
            break

print(f"\n{'=' * 70}")
print(f"Training complete. Best validation IoU: {best_iou:.4f}")

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
er = range(1, len(history['train_loss']) + 1)

axes[0,0].plot(er, history['train_loss'], 'b-', lw=2, label='Train')
axes[0,0].plot(er, history['val_loss'], 'r-', lw=2, label='Validation')
axes[0,0].set_title('Loss', fontweight='bold')
axes[0,0].set_xlabel('Epoch')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(er, history['val_iou'], 'g-', lw=2, label='Val IoU')
best_epoch = np.argmax(history['val_iou']) + 1
axes[0,1].axvline(best_epoch, color='red', ls='--', alpha=0.5,
                  label=f'Best: epoch {best_epoch}')
axes[0,1].axhline(0.90, color='orange', ls=':', label='IoU = 0.90')
axes[0,1].set_title('Validation IoU', fontweight='bold')
axes[0,1].set_xlabel('Epoch')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(er, history['val_dice'], 'c-', lw=2, label='Val Dice')
axes[1,0].set_title('Validation Dice', fontweight='bold')
axes[1,0].set_xlabel('Epoch')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(er, history['lr'], 'm-', lw=2)
axes[1,1].set_title('Learning Rate', fontweight='bold')
axes[1,1].set_xlabel('Epoch')
axes[1,1].set_yscale('log')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Quantum SAM Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/visualizations/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Test Set Evaluation

The best checkpoint (highest validation IoU) is loaded and evaluated on the held-out test set. An optimal threshold is searched on the validation set before computing test metrics.

In [ ]:
# 7.1 - Load best checkpoint
ckpt = torch.load('models/best/best_quantum_sam.pth')
model.load_state_dict(ckpt['model_state'], strict=False)
model.eval()
print(f"Loaded best checkpoint — val IoU: {ckpt['best_iou']:.4f} at epoch {ckpt['epoch']}")

In [ ]:
# 7.2 - Search optimal threshold on validation set
print("Searching optimal threshold on validation set...")
all_preds_val, all_masks_val = [], []

with torch.no_grad():
    for images, masks, boxes, centers in tqdm(val_loader, desc='Val', leave=False):
        images = images.to(DEVICE)
        boxes = boxes.to(DEVICE)
        pred = model(images, boxes=boxes)
        pred = torch.sigmoid(pred).cpu()
        all_preds_val.append(pred)
        all_masks_val.append(masks)

all_preds_val = torch.cat(all_preds_val)
all_masks_val = torch.cat(all_masks_val)

best_thresh, best_thresh_iou = 0.5, 0
for t in np.arange(0.30, 0.70, 0.02):
    pred_bin = (all_preds_val > t).float()
    inter = (pred_bin * all_masks_val).sum()
    union = (pred_bin + all_masks_val).clamp(0, 1).sum()
    iou = ((inter + 1e-6) / (union + 1e-6)).item()
    if iou > best_thresh_iou:
        best_thresh_iou = iou
        best_thresh = t

print(f"Optimal threshold: {best_thresh:.2f} (val IoU: {best_thresh_iou:.4f})")

In [ ]:
# 7.3 - Evaluate on test set
test_ds = SAMSkinDataset(
    CONFIG['seg_images'], CONFIG['seg_masks'], test_files,
    image_size=CONFIG['image_size'], transform=None
)
test_loader = DataLoader(
    test_ds, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=4, pin_memory=True
)

all_ious, all_dices = [], []

with torch.no_grad():
    for images, masks, boxes, centers in tqdm(test_loader, desc='Test'):
        images = images.to(DEVICE)
        boxes = boxes.to(DEVICE)
        pred = model(images, boxes=boxes)
        pred = torch.sigmoid(pred).cpu()

        pred_bin = (pred > best_thresh).float()
        inter = (pred_bin * masks).sum(dim=(1, 2, 3))
        union = (pred_bin + masks).clamp(0, 1).sum(dim=(1, 2, 3))
        iou = (inter + 1e-6) / (union + 1e-6)
        dice = (2 * inter + 1e-6) / (pred_bin.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3)) + 1e-6)

        all_ious.extend(iou.tolist())
        all_dices.extend(dice.tolist())

print(f"\n{'=' * 60}")
print(f"QUANTUM SAM TEST RESULTS")
print(f"{'=' * 60}")
print(f"  IoU:         {np.mean(all_ious):.4f} (±{np.std(all_ious):.4f})")
print(f"  Dice:        {np.mean(all_dices):.4f}")
print(f"  IoU min/max: {np.min(all_ious):.4f} / {np.max(all_ious):.4f}")
print(f"{'=' * 60}")

## 8. Visualization

In [ ]:
# 8.1 - IoU distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(all_ious, bins=30, color='#5B5BD6', edgecolor='white', alpha=0.85)
axes[0].axvline(np.mean(all_ious), color='red', ls='--', lw=2,
                label=f'Mean: {np.mean(all_ious):.4f}')
axes[0].set_title('Test IoU Distribution', fontweight='bold')
axes[0].set_xlabel('IoU')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(all_ious, patch_artist=True,
                boxprops=dict(facecolor='#5B5BD6', alpha=0.7))
axes[1].axhline(0.90, color='green', ls=':', lw=2, label='IoU = 0.90')
axes[1].set_title('Test IoU Box Plot', fontweight='bold')
axes[1].set_ylabel('IoU')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/visualizations/test_iou_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.2 - Sample predictions on test set
fig, axes = plt.subplots(4, 3, figsize=(15, 20))
indices = np.random.choice(len(test_ds), 4, replace=False)

for row, idx in enumerate(indices):
    img, mask, box, center = test_ds[idx]
    with torch.no_grad():
        pred = model(
            img.unsqueeze(0).to(DEVICE),
            boxes=box.unsqueeze(0).to(DEVICE)
        )
    pred = torch.sigmoid(pred).cpu().squeeze().numpy()
    pred_bin = (pred > best_thresh).astype(np.float32)
    mask_np = mask.squeeze().numpy()

    inter = (pred_bin * mask_np).sum()
    union = np.clip(pred_bin + mask_np, 0, 1).sum()
    iou = (inter + 1e-6) / (union + 1e-6)

    # Downscale image to match mask resolution for visualization
    img_small = F.interpolate(
        img.unsqueeze(0), size=(256, 256), mode='bilinear'
    ).squeeze().permute(1, 2, 0).numpy()

    axes[row, 0].imshow(img_small)
    axes[row, 0].set_title('Input image')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(mask_np, cmap='gray')
    axes[row, 1].set_title('Ground truth')
    axes[row, 1].axis('off')
    axes[row, 2].imshow(pred_bin, cmap='gray')
    axes[row, 2].set_title(f'Prediction (IoU: {iou:.3f})')
    axes[row, 2].axis('off')

plt.suptitle('Quantum SAM Predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/predictions/sample_predictions.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Artifacts

In [ ]:
# Save results to Drive for cross-model comparison
import json
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/skin_cancer_project'
os.makedirs(save_dir, exist_ok=True)

shutil.copy('models/best/best_quantum_sam.pth', save_dir)

viz_dir = os.path.join(save_dir, 'outputs')
os.makedirs(viz_dir, exist_ok=True)
import glob
for f in glob.glob('outputs/**/*.png', recursive=True):
    shutil.copy(f, viz_dir)

results = {
    'model': 'Quantum SAM (SAM ViT-B + QCA)',
    'best_val_iou': float(best_iou),
    'test_iou_mean': float(np.mean(all_ious)),
    'test_iou_std': float(np.std(all_ious)),
    'test_dice': float(np.mean(all_dices)),
    'optimal_threshold': float(best_thresh),
    'config': {
        'image_size': CONFIG['image_size'],
        'epochs': CONFIG['epochs'],
        'lr': CONFIG['lr'],
        'encoder_lr': CONFIG['encoder_lr'],
        'unfreeze_blocks': CONFIG['unfreeze_blocks'],
        'n_qubits': CONFIG['n_qubits'],
        'q_layers': CONFIG['q_layers'],
        'seed': SEED,
    }
}
with open(os.path.join(save_dir, 'quantum_sam_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f"Artifacts saved to: {save_dir}")